In [14]:
import json
import ast
import numpy as np
import pandas as pd
import networkx as nx

from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm


In [ ]:
BASE_KG_PATH = "/path/unified_disease_symptom.json" # This file contains the diseases and their possible list of symptoms in dictionary format
MIN_SCORES_PATH = "/path/mapped_data/min_scores.json" # THis file contains minimum weights according to each disease
WTS_PATH = "/path/unified_wts.csv" # This file contains the weights for the unified KG for symptom disease relations
EMBEDDINGS_PATH = "/data/medclip_image_embeddings.npz" # This contains all the image embeddings

df_train = pd.read_csv(
    "/path/ours_new_train.csv"
)
df_test = pd.read_csv(
    "/path/ours_new_test.csv"
)

with open(BASE_KG_PATH) as f:
    disease_symptoms = json.load(f)

with open(MIN_SCORES_PATH) as f:
    min_scores = json.load(f)

df_wts = pd.read_csv(WTS_PATH)

emb_npz = np.load(EMBEDDINGS_PATH, allow_pickle=True)

global_min = min(min_scores.values())


In [19]:
G_base = nx.DiGraph()

for disease, symptoms in disease_symptoms.items():
    disease = str(disease).strip().lower()
    base_wt = min_scores.get(disease, global_min)

    G_base.add_node(disease, type="disease")

    for s in symptoms:
        s = str(s).strip().lower()
        G_base.add_node(s, type="symptom")
        G_base.add_edge(s, disease, weight=base_wt)

for _, row in df_wts.iterrows():
    d = str(row["disease"]).strip().lower()
    s = str(row["symptom"]).strip().lower()
    w = float(row["P(D/S)"])

    if G_base.has_edge(s, d):
        G_base[s][d]["weight"] = w


In [20]:
import ast
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity


def run_case(row,G_base,df_train,emb_npz,alpha=0.1,top_k=3,image_weight_scale=1.0):
    G = nx.DiGraph()
    user = f"USER_{row.id}"
    G.add_node(user, type="user")
    symptoms = ast.literal_eval(row.mapped_all_symptoms)
    if not isinstance(symptoms, list) or len(symptoms) == 0:
        return None, 0.0, []
    symptoms = [str(s).strip().lower() for s in symptoms]
    valid_symptoms = []
    for s in symptoms:
        if s in G_base:
            G.add_node(s, type="symptom")
            G.add_edge(user, s, weight=1.0)
            valid_symptoms.append(s)
    if not valid_symptoms:
        return None, 0.0, []
    candidate_diseases = set()
    for s in valid_symptoms:
        for d in G_base.successors(s):
            if G_base.nodes[d].get("type") != "disease":
                continue
            wt = G_base[s][d]["weight"]
            G.add_node(d, type="disease")
            G.add_edge(s, d, weight=wt)
            candidate_diseases.add(d)
    candidate_diseases = list(candidate_diseases)
    if not candidate_diseases:
        return None, 0.0, []
    disease_space_before_image = candidate_diseases.copy()
    query_img = row.image_to_fetch
    img_type = row.image_type
    if query_img in emb_npz:
        q_emb = emb_npz[query_img].reshape(1, -1)
        df_img = df_train[
            (df_train.image_type == img_type) &
            (df_train.mapped_disease.isin(candidate_diseases))]
        best_sim = -1.0
        best_disease = None
        for _, r in df_img.iterrows():
            img_name = r.image_to_fetch
            if img_name not in emb_npz:
                continue
            emb = emb_npz[img_name].reshape(1, -1)
            sim = cosine_similarity(q_emb, emb)[0][0]
            if sim > best_sim:
                best_sim = sim
                best_disease = str(r.mapped_disease).strip().lower()
        if best_disease is not None:
            G.add_node(query_img, type="image")
            G.add_edge(user, query_img, weight=1.0)
            G.add_edge(
                query_img,
                best_disease,
                weight=best_sim * image_weight_scale
            )

    personalization = {user: 1.0}
    scores = nx.pagerank(
        G,
        alpha=alpha,
        personalization=personalization,
        weight="weight"
    )
    disease_scores = {
        n: s for n, s in scores.items()
        if G.nodes[n].get("type") == "disease"
    }
    if not disease_scores:
        return None, 0.0, disease_space_before_image
    pred = max(disease_scores, key=disease_scores.get)
    conf = disease_scores[pred]
    return pred, conf, disease_space_before_image

In [21]:
preds = []
confs = []
disease_spaces = []

for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
    p, c, ds = run_case(row, G_base, df_train, emb_npz)
    preds.append(p)
    confs.append(c)
    disease_spaces.append(ds)

df_test["algo_prediction"] = preds
df_test["algo_confidence"] = confs
df_test["candidate_diseases"] = disease_spaces


100%|█████████████████████████████████████████████████████████████████████████████████████████| 94/94 [00:02<00:00, 46.20it/s]


In [23]:
correct = 0
total = 0

for _, r in df_test.iterrows():
    if str(r.mapped_disease).strip().lower() == str(r.algo_prediction).strip().lower():
        correct += 1
    total += 1

print(f"Accuracy: {correct/total:.4f} ")

Accuracy: 21.0600 
